# Beca 18 RAG Chatbot

This notebook builds an end-to-end RAG pipeline over the official Beca 18 regulation PDF. The chatbot answers only from retrieved source fragments and refuses to answer when the document does not contain enough evidence.

## Step 0 - Setup

Install dependencies, load the Gemini API key from `.env`, and print package versions.

In [9]:
from pathlib import Path
import os
import re
import time
import textwrap
import importlib.metadata as metadata
from typing import List, Dict, Any

import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
import tiktoken
from tqdm.auto import tqdm

cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
DATA_DIR = PROJECT_ROOT / "data"
PDF_PATH = DATA_DIR / "beca18_reglamento.pdf"
CHROMA_PATH = PROJECT_ROOT / "chroma_db_beca18"

load_dotenv(PROJECT_ROOT / ".env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
if GEMINI_API_KEY:
    print("GEMINI_API_KEY loaded from .env.")
else:
    print("GEMINI_API_KEY not found. Offline extraction/chunking cells can run, but embeddings and generation will be skipped.")

packages = [
    "pypdf",
    "tiktoken",
    "langchain-text-splitters",
    "google-genai",
    "chromadb",
    "ipywidgets",
    "tqdm",
    "python-dotenv",
]
versions = {pkg: metadata.version(pkg) for pkg in packages}
versions

GEMINI_API_KEY loaded from .env.


{'pypdf': '6.11.0',
 'tiktoken': '0.12.0',
 'langchain-text-splitters': '1.1.2',
 'google-genai': '2.2.0',
 'chromadb': '1.5.9',
 'ipywidgets': '8.1.8',
 'tqdm': '4.67.3',
 'python-dotenv': '1.2.2'}

## Step 1 - PDF text extraction

Extract text page by page, add `[PAGE N]` markers, lightly clean whitespace, and print total character and word counts.

In [10]:
def clean_page_text(text: str) -> str:
    text = text or ""
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


reader = PdfReader(str(PDF_PATH))
page_texts = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = clean_page_text(page.extract_text())
    page_texts.append(f"[PAGE {page_number}]\n{page_text}")

document_text = "\n\n".join(page_texts)
char_count = len(document_text)
word_count = len(re.findall(r"\b\w+\b", document_text))
print(f"Pages: {len(page_texts)}")
print(f"Total characters: {char_count:,}")
print(f"Total words: {word_count:,}")
print(document_text[:1000])

Pages: 138
Total characters: 366,398
Total words: 57,281
[PAGE 1]
Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS: El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ de la Oficina de Asesoría Jurídica, y; CONSIDERANDO: Que, la Ley N° 29837 crea el Programa Nacional de Becas y Crédito Educativo (en adelante, el PRONABEC), a cargo del Ministerio de Educación, encargado del diseño, planificación, gestión, monitoreo y evaluación de becas y créditos educativos para el financiamiento de estudios de educación técnica y superior; estudios relacionados con los idiomas, desde la etapa de educación básica, en instituciones técnicas, universitarias y otros centros de formación en general, formen pa

## Step 2 - Tokenization and chunking justification

Gemini embedding requests support an 8,192-token input limit. A 400-token chunk with 60-token overlap is small enough to preserve retrieval precision, large enough to carry legal context, and safely below the embedding limit. The overlap reduces boundary loss when requirements, obligations, or sanctions are split across pages or paragraphs.

In [11]:
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(encoding.encode(text))


total_tokens = count_tokens(document_text)
print(f"Total tokens with cl100k_base: {total_tokens:,}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "],
    length_function=count_tokens,
)

raw_chunks = splitter.split_text(document_text)

def page_from_chunk(text: str):
    match = re.search(r"\[PAGE (\d+)\]", text)
    return int(match.group(1)) if match else None


chunks = []
for idx, text in enumerate(raw_chunks):
    chunks.append(
        {
            "id": f"beca18-{idx:04d}",
            "text": text,
            "metadata": {
                "document": "Resolucion Directoral Ejecutiva N. 033-2026-MINEDU/VMGI-PRONABEC",
                "topic": "Beca 18 regulations",
                "language": "Spanish",
                "page": page_from_chunk(text) or -1,
            },
        }
    )

avg_chars = sum(len(chunk["text"]) for chunk in chunks) / len(chunks)
print(f"Total chunks: {len(chunks):,}")
print(f"Average chunk length: {avg_chars:.1f} characters")
print(chunks[0])

Total tokens with cl100k_base: 102,819
Total chunks: 491
Average chunk length: 788.4 characters
{'id': 'beca18-0000', 'text': '[PAGE 1]', 'metadata': {'document': 'Resolucion Directoral Ejecutiva N. 033-2026-MINEDU/VMGI-PRONABEC', 'topic': 'Beca 18 regulations', 'language': 'Spanish', 'page': 1}}


## Step 3 - Embeddings

Use `gemini-embedding-001` with task-specific embedding modes and exponential backoff for free-tier rate limits.

In [12]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-2.5-flash"


def _embed_with_retry(texts: List[str], task_type: str, batch_size: int = 16, max_retries: int = 6) -> List[List[float]]:
    if genai_client is None:
        raise ValueError("GEMINI_API_KEY is required for embeddings. Create .env from .env.example.")
    embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc=f"Embedding {task_type}"):
        batch = texts[start : start + batch_size]
        for attempt in range(max_retries):
            try:
                response = genai_client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(
                        task_type=task_type,
                        output_dimensionality=768,
                    ),
                )
                embeddings.extend([item.values for item in response.embeddings])
                break
            except Exception as exc:
                if attempt == max_retries - 1:
                    raise
                sleep_seconds = min(60, 2 ** attempt)
                print(f"Embedding retry after error: {exc}. Sleeping {sleep_seconds}s.")
                time.sleep(sleep_seconds)
        time.sleep(1.1)
    return embeddings


def embed_documents(texts: List[str]) -> List[List[float]]:
    return _embed_with_retry(texts, task_type="RETRIEVAL_DOCUMENT")


def embed_query(text: str) -> List[float]:
    return _embed_with_retry([text], task_type="RETRIEVAL_QUERY", batch_size=1)[0]

## Step 4 - Vector database

Create a persistent ChromaDB collection with cosine distance and idempotent indexing.

In [13]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = chroma_client.get_or_create_collection(
    name="beca18_reglamento",
    metadata={"hnsw:space": "cosine"},
)

existing_count = collection.count()
if existing_count > 0:
    print(f"Collection already has {existing_count:,} documents. Skipping embedding.")
elif genai_client is None:
    print("Collection is empty, but GEMINI_API_KEY is missing. Skipping embedding/indexing.")
else:
    texts = [chunk["text"] for chunk in chunks]
    ids = [chunk["id"] for chunk in chunks]
    metadatas = [chunk["metadata"] for chunk in chunks]
    embeddings = embed_documents(texts)
    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings)
    print("Indexing completed.")

print(f"Stored documents: {collection.count():,}")

Collection already has 491 documents. Skipping embedding.
Stored documents: 491


## Step 5 - Semantic search

Query ChromaDB with a Gemini query embedding and return text, metadata, and distance.

In [14]:
def semantic_search(question: str, k: int = 5) -> List[Dict[str, Any]]:
    question_embedding = embed_query(question)
    result = collection.query(
        query_embeddings=[question_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for text, metadata_item, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({"text": text, "metadata": metadata_item, "distance": distance})
    return hits


sample_question = "Cuales son los requisitos para postular a Beca 18?"
if genai_client is None or collection.count() == 0:
    print("Skipping semantic search test until GEMINI_API_KEY is available and the collection is indexed.")
else:
    sample_hits = semantic_search(sample_question, k=3)
    for idx, hit in enumerate(sample_hits, start=1):
        print(f"\nResult {idx} | distance={hit['distance']:.4f} | page={hit['metadata'].get('page')}")
        print(textwrap.shorten(hit["text"].replace("\n", " "), width=500))

Embedding RETRIEVAL_QUERY:   0%|          | 0/1 [00:00<?, ?it/s]


Result 1 | distance=0.1964 | page=-1
16 N° REQUISITO DOCUMENTO DE ACREDITACIÓN o FORMA DE ACREDITACIÓN documento oficial de la IES donde se acredite que mantiene dicha vacante. 2 Haber culminado el nivel secundario de la Educación Básica Regular (EBR) o Alternativa (EBA) o Especial (EBE). Los estudios deben ser reconocidos por el Ministerio de Educación. *En el caso de la Beca 18 Ordinaria el postulante debe haber egresado de la Educación Básica Regular (EBR) o Básica Alternativa (EBA), como máximo en los tres (3) años [...]

Result 2 | distance=0.2180 | page=-1
. 3 Acreditar alto rendimiento académico: Para la Beca 18 Ordinaria, Beca Huallaga, Beca VRAEM y BEAHD: acreditar tercio superior en los dos últimos grados concluidos de secundaria EBR o EBA o EBE. Para la Beca Protección, Beca CNA, Beca PA, Beca EIB, Beca FF.AA.: acreditar medio superior en los dos últimos grados concluidos de secundaria de EBR o EBA o EBE. • El Módulo de Postulación para la Selección, con base en la informac

## Step 6 - Grounded generation

Generate answers using only retrieved context. If the context is insufficient, the answer must decline.

In [15]:
SYSTEM_PROMPT = """
You are a grounded assistant for the official Beca 18 regulation.
Answer exclusively from the retrieved context.
Cite page numbers when they are available.
If the retrieved context is insufficient, respond exactly:
"The document does not contain information about this topic."
Do not use outside knowledge.
"""


def format_context(hits: List[Dict[str, Any]]) -> str:
    blocks = []
    for idx, hit in enumerate(hits, start=1):
        page = hit["metadata"].get("page", "unknown")
        blocks.append(
            f"[SOURCE {idx} | PAGE {page} | DISTANCE {hit['distance']:.4f}]\n{hit['text']}"
        )
    return "\n\n".join(blocks)


def answer_with_context(question: str, k: int = 5) -> Dict[str, Any]:
    if genai_client is None:
        raise ValueError("GEMINI_API_KEY is required for grounded generation. Create .env from .env.example.")
    hits = semantic_search(question, k=k)
    context = format_context(hits)
    prompt = f"Question: {question}\n\nRetrieved context:\n{context}\n\nAnswer:"
    response = genai_client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.0,
        ),
    )
    return {"answer": response.text, "sources": hits}


test_questions = [
    "Cuales son los requisitos de elegibilidad para Beca 18?",
    "Que modalidades de beca se mencionan?",
    "Cual es el monto de la subvencion mensual?",
    "Cuales son las obligaciones de los estudiantes becarios?",
    "En que condiciones se puede perder la beca?",
    "Cual es la mejor receta para preparar ceviche?",
]

if genai_client is None or collection.count() == 0:
    print("Skipping grounded generation tests until GEMINI_API_KEY is available and the collection is indexed.")
else:
    for question in test_questions:
        result = answer_with_context(question, k=5)
        print("\n" + "=" * 100)
        print("QUESTION:", question)
        print("ANSWER:", result["answer"])

Embedding RETRIEVAL_QUERY:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: Cuales son los requisitos de elegibilidad para Beca 18?
ANSWER: La Beca 18 agrupa diez (10) becas de pregrado y especiales, y cada una se sujeta a sus propias normas y condiciones (SOURCE 1, SOURCE 4, SOURCE 5).

Para la **Beca 18 Ordinaria**, los requisitos de elegibilidad son:
*   **Edad:** Ser menor de 22 años a la fecha de la publicación de la Norma Técnica (SOURCE 2).
*   **Condición de vulnerabilidad:** Clasificación como pobre o pobre extremo según los criterios de focalización establecidos por el Sistema de Focalización de Hogares (SISFOH) (SOURCE 2).
*   **Rendimiento académico:** Tercio superior en los dos últimos grados concluidos de secundaria EBR o EBE o su equivalente en EBA (SOURCE 2).
*   **Nivel secundario:** Haber culminado el nivel secundario de la Educación Básica Regular (EBR) o Alternativa (EBA) o Especial (EBE), reconocidos por el Ministerio de Educación (SOURCE 3).
*   **Año de egreso (para Beca 18 Ordinaria):** Haber egresado de la Educación Básica R

Embedding RETRIEVAL_QUERY:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: Que modalidades de beca se mencionan?
ANSWER: Las modalidades de beca mencionadas en el documento son las siguientes:

*   Beca EIB (Beca de Formación en Educación Intercultural Bilingüe) (Fuente 1, 2, 3, 4)
*   BEAHD (Beca Excelencia Académica para Hijos de Docentes) (Fuente 1, 2, 3, 4, 5)
*   Beca FF.AA. (Beca para Licenciados del Servicio Militar Voluntario) (Fuente 1, 2, 3, 4)
*   Beca Huallaga (Beca para pobladores residentes en el Huallaga) (Fuente 1, 2, 3, 4, 5)
*   Beca 18 (ordinaria) (Fuente 2, 3, 4)
*   Beca Protección (Beca para adolescentes con protección estatal) (Fuente 2, 3, 4)
*   Beca CNA (Beca para Comunidades Nativas Amazónicas) (Fuente 2, 3, 4)
*   Beca VRAEM (Beca para pobladores residentes del valle de los ríos Apurímac, Ene y Mantaro) (Fuente 2, 3, 4, 5)
*   Beca PA (Beca para Pueblo Afroperuano) (Fuente 2, 3, 4, 5)
*   Beca REPARED (Beca para víctimas de la violencia habida en el país durante los años 1980 – 2000) (Fuente 2, 3, 4, 5)


Embedding RETRIEVAL_QUERY:   0%|          | 0/1 [00:00<?, ?it/s]

...


QUESTION: Cuales son las obligaciones de los estudiantes becarios?
ANSWER: Las obligaciones de los estudiantes becarios se rigen de acuerdo con la Ley N° 29837 y su Reglamento, así como las normas internas del Programa y la normativa vigente del PRONABEC (Fuente 1, 19.1; Fuente 2, g; Fuente 5, 8.8).

Como parte de sus obligaciones, los becarios deben:
*   Cumplir con el “Compromiso del Servicio al Perú”, para revertir a favor del país los beneficios de la capacitación recibida por el Estado (Anexo N° 08), conforme a lo estipulado en la Ley N° 29837 y su Reglamento (Fuente 1, 19.3).
*   Participar en las acciones de acompañamiento académico, socioemocional y de bienestar que promueva el PRONABEC, durante toda su etapa formativa, para recibir orientación y apoyo para la permanencia y egreso de los estudios superiores de forma óptima (Fuente 2, h).
*   Someterse a las disposiciones del Comité de Becas y de la Dirección de Gestión de Becas del PRONABEC, según corresponda a sus competencia

Embedding RETRIEVAL_QUERY:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: En que condiciones se puede perder la beca?
ANSWER: Se puede perder la beca en las siguientes condiciones:

*   Si se verifica que el postulante no cumple con alguno de los requisitos después de ser declarado becario, se procederá conforme a lo establecido en el artículo 34.3 de la Ley N° 27444. (SOURCE 1, 12.25)
*   Si se detecta falsedad en la información por parte del declarante, él o ella asumirá las consecuencias legales correspondientes. (SOURCE 3, QUINTA)
*   El incumplimiento de cualquiera de las estipulaciones, condiciones y/o requisitos establecidos en las Bases es considerado causal para declarar al becario NO APTO o declarar la nulidad de la adjudicación de la beca. (SOURCE 3, QUINTA)
*   En caso de desaprobación de materias. (SOURCE 5, i)
*   Por incumplir otras obligaciones como becario. (SOURCE 5, i)
*   Por presentar información o documentación falsa con la finalidad de continuar con la beca. (SOURCE 5, i)


Embedding RETRIEVAL_QUERY:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: Cual es la mejor receta para preparar ceviche?
ANSWER: The document does not contain information about this topic.


## Step 7 - Interactive chat interface

Ask questions, control top-k retrieval, and inspect expandable source fragments.

In [16]:
question_box = widgets.Text(
    value="",
    placeholder="Escribe una pregunta sobre Beca 18",
    description="Pregunta:",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask", button_style="primary")
clear_button = widgets.Button(description="Clear")
k_slider = widgets.IntSlider(value=5, min=1, max=10, step=1, description="k")
output_area = widgets.Output()


def render_sources(hits):
    children = []
    titles = []
    for idx, hit in enumerate(hits, start=1):
        page = hit["metadata"].get("page", "unknown")
        distance = hit["distance"]
        source_output = widgets.Output()
        with source_output:
            print(hit["text"])
        children.append(source_output)
        titles.append(f"Source {idx} | page {page} | distance {distance:.4f}")
    accordion = widgets.Accordion(children=children)
    for idx, title in enumerate(titles):
        accordion.set_title(idx, title)
    return accordion


def on_ask(_):
    question = question_box.value.strip()
    if not question:
        return
    with output_area:
        clear_output()
        print("Searching and generating answer...")
        result = answer_with_context(question, k=k_slider.value)
        clear_output()
        display(Markdown(result["answer"]))
        display(render_sources(result["sources"]))


def on_clear(_):
    question_box.value = ""
    with output_area:
        clear_output()


ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

if genai_client is None or collection.count() == 0:
    display(Markdown("Create `.env` with `GEMINI_API_KEY`, rerun the notebook, and the chat interface will answer from indexed sources."))

display(widgets.VBox([
    widgets.HBox([question_box, ask_button, clear_button]),
    k_slider,
    output_area,
]))